# Churn — HistGradientBoosting with Optuna Optimization

## 0. Configuration

All tuneable settings live here — change them before running the rest of the notebook.

In [ ]:
from datetime import datetime

# ── Data settings ────────────────────────────────────────────────────────
DATA_CONFIG = {
    "data_path": r"data/churn_targets.parquet",
    "target": "CHURN_2M",
    "drop_cols": [
        "CARD_ID", "CARD_CANCELLATION_DATE", "CHURN_1M", "CHURN_3M",
        "STATUS", "BIRTH_DT", "OPEN_DT", "MONTH_END", "CUST_ID",
        "LAST_EXPOSURE_DATE", "MONTH_YYYYMM",
    ],
    "random_state": 42,
    "test_month_start": datetime(2025, 2, 1),
    "month_window_start": datetime(2022, 2, 1),
    "month_window_end": datetime(2025, 1, 31),
    "majority_reduction_pct": 0.10,
    "n_lags": 12,
    "priority_non_null_cols": [
        "REMAINING_INSTALLMENT_AMOUNT_lag1",
        "REMAINING_INSTALLMENT_COUNT_lag1",
        "TOTAL_EXPOSURE_AMT_1M_lag1",
        "STMT_PYMNT_AMT_lag1",
        "AVG_TICKET_MTH_lag1",
        "MCC_MOST_FREQUENT_MTH_lag1",
    ],
    "use_streaming_collect": True,
}

# ── Optuna settings ──────────────────────────────────────────────────────
OPTUNA_CONFIG = {
    "n_trials": 100,
    "cv_folds": 5,
    "optimization_metric": "avg_precision",   # PR-AUC — best single metric for imbalanced
    "direction": "maximize",
    "timeout": 7200,                          # 2-hour safety timeout (seconds)
    "n_startup_trials": 15,                   # random exploration before TPE kicks in
    "pruning": True,
}

# ── MLflow settings ──────────────────────────────────────────────────────
EXPERIMENT = "CHURN-2M"
RUN_NAME = "hgb-optuna"

# ── Model save path (for explainability) ─────────────────────────────────
MODEL_SAVE_DIR = "training/saved_models/hist_gradient_boosting"

In [ ]:
import os, mlflow

from mlflow_utils import TRACKING_URI
 
print("ENV MLFLOW_TRACKING_URI:", os.environ.get("MLFLOW_TRACKING_URI"))

print("mlflow_utils.TRACKING_URI:", TRACKING_URI)

mlflow.set_tracking_uri(TRACKING_URI)

print("mlflow.get_tracking_uri():", mlflow.get_tracking_uri())
 

## 1. Imports & Setup

In [ ]:
import logging
import os
import sys
import tempfile
import time

import numpy as np
import optuna
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

ROOT = os.path.abspath(".")
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "training"))

from data_processing import load_and_prepare_data
from mlflow_utils import (
    configure_training_logging,
    log_classification_results,
    log_dataset_info,
    log_feature_importance,
    save_model_local,
    start_run,
)

configure_training_logging()
log = logging.getLogger("churn")
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. Load & Prepare Data

In [ ]:
X_train_pd, X_test_pd, y_train, y_test, feature_names, data_info = (
    load_and_prepare_data(DATA_CONFIG)
)

print(f"X_train: {X_train_pd.shape}  X_test: {X_test_pd.shape}")
print(f"Features: {len(feature_names)}  scale_pos_weight: {data_info['scale_pos_weight']}")

## 3. Optuna Hyperparameter Optimization

Optimizes PR-AUC (Average Precision) via **expanding-window temporal CV** —
trains on earlier months and validates on later months to prevent leakage.
PR-AUC is the most informative threshold-independent metric for highly
imbalanced churn prediction.

In [ ]:
def temporal_expanding_cv(anchor_months, n_splits=5):
    """Expanding-window temporal CV that prevents future-to-past leakage.

    The last *n_splits* distinct anchor months become validation months.
    For each, training includes only cards with strictly earlier anchor months.
    """
    unique_months = np.sort(np.unique(anchor_months))
    if len(unique_months) <= n_splits:
        raise ValueError(
            f"Only {len(unique_months)} unique months; need > {n_splits}"
        )
    val_months = unique_months[-n_splits:]
    splits = []
    for vm in val_months:
        train_idx = np.where(anchor_months < vm)[0]
        val_idx = np.where(anchor_months == vm)[0]
        if len(train_idx) > 0 and len(val_idx) > 0:
            splits.append((train_idx, val_idx))
    return splits


anchor_months = data_info["_train_anchor_months"]
cv_splits = temporal_expanding_cv(anchor_months, n_splits=OPTUNA_CONFIG["cv_folds"])

log.info(
    "Temporal CV: %d folds | validation months: %s",
    len(cv_splits),
    np.sort(np.unique(anchor_months))[-OPTUNA_CONFIG["cv_folds"]:],
)
for i, (tr, va) in enumerate(cv_splits):
    pos_tr = y_train[tr].sum()
    pos_va = y_train[va].sum()
    log.info(
        "  fold %d — train: %d (%.2f%% pos)  val: %d (%.2f%% pos)",
        i, len(tr), 100 * pos_tr / len(tr), len(va), 100 * pos_va / len(va),
    )


def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "max_iter": trial.suggest_int("max_iter", 200, 2000, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 255),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 200),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-8, 10.0, log=True),
        "max_bins": trial.suggest_int("max_bins", 32, 255),
        "early_stopping": True,
        "n_iter_no_change": trial.suggest_int("n_iter_no_change", 10, 50),
        "validation_fraction": trial.suggest_float("validation_fraction", 0.05, 0.15),
        "random_state": DATA_CONFIG["random_state"],
    }
    use_balanced_weights = trial.suggest_categorical("use_balanced_weights", [True, False])

    fold_scores = []
    for fold_idx, (train_idx, val_idx) in enumerate(cv_splits):
        X_tr = X_train_pd.iloc[train_idx]
        X_val = X_train_pd.iloc[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        clf = HistGradientBoostingClassifier(**params)
        fit_kw = {}
        if use_balanced_weights:
            fit_kw["sample_weight"] = compute_sample_weight("balanced", y_tr)

        clf.fit(X_tr, y_tr, **fit_kw)
        y_proba_val = clf.predict_proba(X_val)[:, 1]
        fold_scores.append(average_precision_score(y_val, y_proba_val))

        trial.report(np.mean(fold_scores), fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(fold_scores)


sampler = optuna.samplers.TPESampler(
    seed=DATA_CONFIG["random_state"],
    n_startup_trials=OPTUNA_CONFIG["n_startup_trials"],
)
pruner = (
    optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
    if OPTUNA_CONFIG["pruning"]
    else optuna.pruners.NopPruner()
)

study = optuna.create_study(
    direction=OPTUNA_CONFIG["direction"],
    sampler=sampler,
    pruner=pruner,
    study_name="hgb-churn-optuna",
)

log.info("Starting Optuna optimization (%s trials)...", OPTUNA_CONFIG["n_trials"])
_t_optuna = time.perf_counter()

study.optimize(
    objective,
    n_trials=OPTUNA_CONFIG["n_trials"],
    timeout=OPTUNA_CONFIG["timeout"],
    show_progress_bar=True,
)

n_complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
log.info(
    "Optuna finished in %.1fs | best PR-AUC: %.4f | completed: %d/%d trials",
    time.perf_counter() - _t_optuna,
    study.best_value,
    n_complete,
    len(study.trials),
)

print(f"\nBest trial #{study.best_trial.number}:")
print(f"  PR-AUC (temporal CV mean): {study.best_value:.4f}")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

best_trial_params = {
    k: v for k, v in study.best_params.items() if k != "use_balanced_weights"
}
best_trial_params["random_state"] = DATA_CONFIG["random_state"]
best_trial_params["early_stopping"] = True
use_balanced = study.best_params["use_balanced_weights"]

best_model = HistGradientBoostingClassifier(**best_trial_params)

fit_kw = {}
if use_balanced:
    fit_kw["sample_weight"] = compute_sample_weight("balanced", y_train)

log.info("Retraining best model on full training set (balanced_weights=%s)...", use_balanced)
_t_retrain = time.perf_counter()
best_model.fit(X_train_pd, y_train, **fit_kw)
log.info("Retrain done in %.1fs  |  n_iter_=%d", time.perf_counter() - _t_retrain, best_model.n_iter_)

model = best_model

# ── Optuna visualizations ────────────────────────────────────────────────
try:
    from optuna.visualization.matplotlib import (
        plot_optimization_history,
        plot_param_importances,
        plot_parallel_coordinate,
    )

    plot_optimization_history(study)
    plt.gcf().set_size_inches(10, 5)
    plt.tight_layout()
    plt.show()

    plot_param_importances(study)
    plt.gcf().set_size_inches(10, 5)
    plt.tight_layout()
    plt.show()

    plot_parallel_coordinate(study)
    plt.gcf().set_size_inches(14, 5)
    plt.tight_layout()
    plt.show()
except Exception as e:
    log.warning("Optuna matplotlib visualization unavailable: %s", e)

# Trial summary table
trials_df = study.trials_dataframe().sort_values("value", ascending=False)
print(f"\nTop 10 trials (by PR-AUC):")
print(trials_df.head(10)[["number", "value", "state", "duration"]].to_string(index=False))

## 4. Permutation Importance (quick diagnostic)

In [ ]:
from sklearn.inspection import permutation_importance

_perm_idx = X_test_pd.sample(n=min(2000, len(X_test_pd)), random_state=42).index
perm = permutation_importance(
    best_model,
    X_test_pd.loc[_perm_idx],
    y_test[_perm_idx],
    n_repeats=3,
    random_state=42,
    scoring="average_precision",
    n_jobs=-1,
)

perm_df = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print(perm_df.head(25).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
plot_df = perm_df.head(20).iloc[::-1]
ax.barh(plot_df["feature"], plot_df["importance_mean"], xerr=plot_df["importance_std"])
ax.set_xlabel("Mean decrease in PR-AUC (permutation)")
ax.set_title("Top 20 Features — Permutation Importance (best Optuna model)")
fig.tight_layout()
plt.show()

## 5. Threshold Tuning

In [ ]:
import mlflow
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    cohen_kappa_score,
    f1_score,
    fbeta_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

log.info("Threshold tuning on best Optuna model...")

y_proba = best_model.predict_proba(X_test_pd)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[0], name="HGB Optuna")
axes[0].set_title("Precision-Recall Curve")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], name="HGB Optuna")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[1].set_title("ROC Curve")

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
precision_t = precision[:-1]
recall_t = recall[:-1]

den = precision_t + recall_t
f1s = np.where(den > 0, 2.0 * precision_t * recall_t / den, 0.0)
# F2 weights recall 2x more — preferable when missing a churner is costlier
f2s = np.where(den > 0, 5.0 * precision_t * recall_t / (4.0 * precision_t + recall_t), 0.0)

best_f1_idx = int(np.nanargmax(f1s))
best_f2_idx = int(np.nanargmax(f2s))
threshold_f1 = float(thresholds[best_f1_idx])
threshold_f2 = float(thresholds[best_f2_idx])

axes[2].plot(thresholds, precision_t, label="Precision", alpha=0.8)
axes[2].plot(thresholds, recall_t, label="Recall", alpha=0.8)
axes[2].plot(thresholds, f1s, label="F1", linewidth=2)
axes[2].plot(thresholds, f2s, label="F2 (recall-heavy)", linewidth=2, linestyle="--")
axes[2].axvline(threshold_f1, color="r", linestyle="--", alpha=0.7,
                label=f"F1 thr = {threshold_f1:.3f}")
axes[2].axvline(threshold_f2, color="g", linestyle="--", alpha=0.7,
                label=f"F2 thr = {threshold_f2:.3f}")
axes[2].set_xlabel("Threshold")
axes[2].set_ylabel("Score")
axes[2].set_title("Metrics vs Decision Threshold")
axes[2].legend(loc="best", fontsize=7)
axes[2].grid(alpha=0.15)

fig.tight_layout()
plt.show()

best_threshold = threshold_f1

print(f"\n{'─' * 65}")
print(f"F1 threshold : {threshold_f1:.4f}  →  P={precision_t[best_f1_idx]:.4f}  "
      f"R={recall_t[best_f1_idx]:.4f}  F1={f1s[best_f1_idx]:.4f}")
print(f"F2 threshold : {threshold_f2:.4f}  →  P={precision_t[best_f2_idx]:.4f}  "
      f"R={recall_t[best_f2_idx]:.4f}  F2={f2s[best_f2_idx]:.4f}")
print(f"Selected     : {best_threshold:.4f}  (max F1)")
print(f"{'─' * 65}")

In [ ]:
y_pred_tuned = (y_proba >= best_threshold).astype(int)

tuned_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_tuned),
    "precision": precision_score(y_test, y_pred_tuned, zero_division=0),
    "recall": recall_score(y_test, y_pred_tuned, zero_division=0),
    "f1": f1_score(y_test, y_pred_tuned, zero_division=0),
    "f2": fbeta_score(y_test, y_pred_tuned, beta=2, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
    "avg_precision": average_precision_score(y_test, y_proba),
    "mcc": matthews_corrcoef(y_test, y_pred_tuned),
    "cohen_kappa": cohen_kappa_score(y_test, y_pred_tuned),
    "brier_score": brier_score_loss(y_test, y_proba),
    "log_loss": log_loss(y_test, y_proba),
    "cv_avg_precision": study.best_value,
}

mlflow_params = {k: str(v) for k, v in study.best_params.items()}
mlflow_params.update({
    "threshold": str(round(best_threshold, 4)),
    "threshold_f2": str(round(threshold_f2, 4)),
    "threshold_selection_metric": "max_f1_from_pr_curve",
    "optuna_n_trials": str(len(study.trials)),
    "optuna_n_completed": str(n_complete),
    "optuna_best_cv_pr_auc": str(round(study.best_value, 6)),
})

with start_run(
    EXPERIMENT,
    run_name=f"{RUN_NAME}-best",
    params=mlflow_params,
) as run_best:
    log_dataset_info(data_info)
    mlflow.log_metrics(tuned_metrics)

    mlflow.sklearn.log_model(best_model, artifact_path="model")
    log_feature_importance(best_model, feature_names)

    with tempfile.TemporaryDirectory() as tmp:
        trials_csv = study.trials_dataframe()
        trials_path = os.path.join(tmp, "optuna_trials.csv")
        trials_csv.to_csv(trials_path, index=False)
        mlflow.log_artifact(trials_path)

        sweep_df = pd.DataFrame({
            "threshold": thresholds,
            "precision": precision_t,
            "recall": recall_t,
            "f1": f1s,
            "f2": f2s,
        })
        sweep_path = os.path.join(tmp, "threshold_sweep.csv")
        sweep_df.to_csv(sweep_path, index=False)
        mlflow.log_artifact(sweep_path)

    fig_cm, axes_cm = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred_tuned, ax=axes_cm[0], cmap="Blues",
    )
    axes_cm[0].set_title(f"Threshold = {best_threshold:.3f} (max F1)")

    y_pred_f2 = (y_proba >= threshold_f2).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred_f2, ax=axes_cm[1], cmap="Greens",
    )
    axes_cm[1].set_title(f"Threshold = {threshold_f2:.3f} (max F2)")
    fig_cm.suptitle("Confusion Matrices — F1 vs F2 threshold", fontsize=13)
    fig_cm.tight_layout()
    plt.show()

    run_id = run_best.info.run_id

print(f"\nTuned metrics (F1 threshold):")
for k, v in tuned_metrics.items():
    print(f"  {k:20s}: {v:.4f}")
print(f"\nRun ID: {run_id}")

## 6. Compare All Runs

In [ ]:
from mlflow_utils import TRACKING_URI

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT)

runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    order_by=["metrics.avg_precision DESC"],
)

key_cols = [
    "tags.mlflow.runName",
    "metrics.avg_precision", "metrics.f1", "metrics.f2",
    "metrics.precision", "metrics.recall",
    "metrics.roc_auc", "metrics.mcc",
    "params.threshold",
]
display_cols = [c for c in key_cols if c in runs_df.columns]
comparison = runs_df[display_cols].copy()
comparison.columns = [c.rsplit(".", 1)[-1] for c in comparison.columns]

print("=== MLflow Run Comparison ===")
print(comparison.to_string(index=False))

## 7. Save Model Locally

Saves the model + full metadata so it can be loaded in the explainability notebook.

In [ ]:
model_metadata = {
    "experiment": EXPERIMENT,
    "run_name": f"{RUN_NAME}-best",
    "run_id": run_id,
    "model_type": "HistGradientBoostingClassifier",
    "hyperparameters": study.best_params,
    "optuna_best_cv_pr_auc": study.best_value,
    "optuna_n_trials": len(study.trials),
    "optuna_n_completed": n_complete,
    "feature_names": feature_names,
    "data_info": {k: v for k, v in data_info.items() if not k.startswith("_")},
    "best_threshold": best_threshold,
    "threshold_f2": threshold_f2,
    "tuned_metrics": tuned_metrics,
}

save_path = save_model_local(best_model, MODEL_SAVE_DIR, model_metadata)
print(f"Model saved to: {save_path}")

## 8. Cleanup

In [ ]:
import gc

log.info("Cleaning up...")
del X_train_pd, X_test_pd, best_model
gc.collect()
log.info("Done.")